# Traffic Demand Prediction (CatBoost + XGBoost Only)
Optimized for macOS without LightGBM

In [1]:

# Install if needed
# !pip install catboost xgboost pygeohash scikit-learn pandas numpy


In [2]:

import pandas as pd
import numpy as np
import pygeohash as pgh

from sklearn.model_selection import KFold
from sklearn.metrics import r2_score

from catboost import CatBoostRegressor
from xgboost import XGBRegressor

import warnings
warnings.filterwarnings('ignore')


In [3]:

train = pd.read_csv('train.csv')
test = pd.read_csv('test.csv')

print('Train:', train.shape)
print('Test:', test.shape)


Train: (77299, 11)
Test: (41778, 10)


In [4]:

def decode_geohash(g):
    try:
        lat, lon = pgh.decode(g)
        return pd.Series([lat, lon])
    except:
        return pd.Series([np.nan, np.nan])

train[['latitude','longitude']] = train['geohash'].apply(decode_geohash)
test[['latitude','longitude']] = test['geohash'].apply(decode_geohash)


In [5]:

def create_features(df):

    df['timestamp'] = pd.to_datetime(df['timestamp'], errors='coerce')

    df['hour'] = df['timestamp'].dt.hour
    df['minute'] = df['timestamp'].dt.minute
    df['dayofweek'] = df['timestamp'].dt.dayofweek

    df['hour_sin'] = np.sin(2*np.pi*df['hour']/24)
    df['hour_cos'] = np.cos(2*np.pi*df['hour']/24)

    df['lat_lon_interaction'] = (
        df['latitude'] * df['longitude']
    )

    return df

train = create_features(train)
test = create_features(test)


In [6]:
# Fill categorical columns
cat_cols = [
    'RoadType',
    'LargeVehicles',
    'Landmarks',
    'Weather'
]

for col in cat_cols:
    train[col] = train[col].fillna("Unknown")
    test[col] = test[col].fillna("Unknown")


# Fill numeric columns only
for col in train.columns:

    if pd.api.types.is_numeric_dtype(train[col]):

        median_value = train[col].median()

        train[col] = train[col].fillna(median_value)

        if col in test.columns:
            test[col] = test[col].fillna(median_value)

In [7]:

drop_cols = [
    'Index',
    'geohash',
    'timestamp'
]

X = train.drop(drop_cols + ['demand'], axis=1)
y = train['demand']

X_test = test.drop(drop_cols, axis=1)

for c in cat_cols:
    X[c] = X[c].astype(str)
    X_test[c] = X_test[c].astype(str)

cat_idx = [X.columns.get_loc(c) for c in cat_cols]

print(X.shape)


(77299, 15)


In [8]:

kf = KFold(
    n_splits=5,
    shuffle=True,
    random_state=42
)

cat_preds = np.zeros(len(X_test))
xgb_preds = np.zeros(len(X_test))

scores = []


In [9]:

for fold,(tr_idx,val_idx) in enumerate(kf.split(X),1):

    print(f'Fold {fold}')

    X_train = X.iloc[tr_idx]
    y_train = y.iloc[tr_idx]

    X_valid = X.iloc[val_idx]
    y_valid = y.iloc[val_idx]

    cat_model = CatBoostRegressor(
        iterations=5000,
        learning_rate=0.03,
        depth=10,
        loss_function='RMSE',
        random_seed=42,
        verbose=False
    )

    cat_model.fit(
        X_train,
        y_train,
        cat_features=cat_idx
    )

    pred = cat_model.predict(X_valid)

    score = r2_score(y_valid, pred)
    scores.append(score)

    print('R2:', score)

    cat_preds += cat_model.predict(X_test) / 5

    X_train_xgb = pd.get_dummies(X_train)
    X_valid_xgb = pd.get_dummies(X_valid)
    X_test_xgb = pd.get_dummies(X_test)

    X_train_xgb, X_valid_xgb = X_train_xgb.align(
        X_valid_xgb,
        join='left',
        axis=1,
        fill_value=0
    )

    X_train_xgb, X_test_xgb = X_train_xgb.align(
        X_test_xgb,
        join='left',
        axis=1,
        fill_value=0
    )

    xgb_model = XGBRegressor(
        n_estimators=2500,
        learning_rate=0.03,
        max_depth=10,
        subsample=0.8,
        colsample_bytree=0.8,
        random_state=42
    )

    xgb_model.fit(X_train_xgb, y_train)

    xgb_preds += xgb_model.predict(X_test_xgb) / 5


Fold 1
R2: 0.943371012621364
Fold 2
R2: 0.9416798106790036
Fold 3
R2: 0.9443867753816844
Fold 4
R2: 0.9370921817597946
Fold 5
R2: 0.9440968766418572


In [10]:

print('Mean CV R2:', np.mean(scores))

final_predictions = (
    0.7 * cat_preds +
    0.3 * xgb_preds
)

submission = pd.DataFrame({
    'Index': test['Index'],
    'demand': final_predictions
})

submission.to_csv('submission.csv', index=False)

print(submission.shape)
submission.head()


Mean CV R2: 0.9421253314167407
(41778, 2)


,Index,demand
0,0,0.087170
1,1,0.054367
2,2,0.027640
3,3,0.022827
4,4,0.050267
